#Task 1. Tf-idf based recommendation

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
genre_columns = [
    "unknown", "Action", "Adventure", "Animation", "Childrens",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]
m_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'imdb_url'] + genre_columns
movies = pd.read_csv('u.item', sep='|', names=m_cols, encoding='latin-1')

#This converts binary genres into a text string for TF-IDF
def get_genre_string(row):
    genres = [genre for genre in genre_columns if row[genre] == 1]
    return " ".join(genres)

movies['genre_str'] = movies.apply(get_genre_string, axis=1)

print("Sample Genre Strings:")
print(movies[['movie_title', 'genre_str']].head())

Sample Genre Strings:
         movie_title                   genre_str
0   Toy Story (1995)  Animation Childrens Comedy
1   GoldenEye (1995)   Action Adventure Thriller
2  Four Rooms (1995)                    Thriller
3  Get Shorty (1995)         Action Comedy Drama
4     Copycat (1995)        Crime Drama Thriller


In [ ]:
# Initialize the TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Construct the TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(movies['genre_str'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

TF-IDF Matrix Shape: (1682, 21)


In [ ]:
# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix calculated.")

Cosine Similarity Matrix calculated.


In [ ]:
# Create a reverse mapping of titles to indices
indices = pd.Series(movies.index, index=movies['movie_title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim):
    #this gets the index of the movie that matches the title
    idx = indices[title]

    #this gets the pairwsie similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    #this gets the scores of the 10 most similar movies (excluding the movie itself)
    sim_scores = sim_scores[1:6]

    #this gets the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar movies
    return movies['movie_title'].iloc[movie_indices]

test_movie = "Toy Story (1995)"
print(f"Recommendations for '{test_movie}':")
print(get_recommendations(test_movie))

Recommendations for 'Toy Story (1995)':
421    Aladdin and the King of Thieves (1996)
101                    Aristocats, The (1970)
403                          Pinocchio (1940)
624            Sword in the Stone, The (1963)
945             Fox and the Hound, The (1981)
Name: movie_title, dtype: object


In [ ]:
test_movie = "Sword in the Stone, The (1963)"
print(f"Recommendations for '{test_movie}':")
print(get_recommendations(test_movie))

Recommendations for 'Sword in the Stone, The (1963)':
403                                Pinocchio (1940)
624                  Sword in the Stone, The (1963)
945                   Fox and the Hound, The (1981)
968     Winnie the Pooh and the Blustery Day (1968)
1065                                   Balto (1995)
Name: movie_title, dtype: object


In [ ]:
test_movie = "Balto (1995)"
print(f"Recommendations for '{test_movie}':")
print(get_recommendations(test_movie))

Recommendations for 'Balto (1995)':
403                                Pinocchio (1940)
624                  Sword in the Stone, The (1963)
945                   Fox and the Hound, The (1981)
968     Winnie the Pooh and the Blustery Day (1968)
1065                                   Balto (1995)
Name: movie_title, dtype: object


In [ ]:
test_movie = "Four Rooms (1995)"
print(f"Recommendations for '{test_movie}':")
print(get_recommendations(test_movie))

Recommendations for 'Four Rooms (1995)':
197    Nikita (La Femme Nikita) (1990)
217                   Cape Fear (1991)
358             Assignment, The (1997)
465               Red Rock West (1992)
594                    Fan, The (1996)
Name: movie_title, dtype: object


In [ ]:
test_movie = "Nikita (La Femme Nikita) (1990)"
print(f"Recommendations for '{test_movie}':")
print(get_recommendations(test_movie))

Recommendations for 'Nikita (La Femme Nikita) (1990)':
197    Nikita (La Femme Nikita) (1990)
217                   Cape Fear (1991)
358             Assignment, The (1997)
465               Red Rock West (1992)
594                    Fan, The (1996)
Name: movie_title, dtype: object


#Task 2: User-Profile-Based Content Recommender

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the ratings data (u.data)
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

# 2. Pivot the data to create the User-Item Matrix
# Rows = Users, Columns = Movies, Values = Ratings
user_item_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating')

# 3. Fill NaN values with 0 (assuming 0 means the user hasn't seen the movie)
user_item_matrix_filled = user_item_matrix.fillna(0)

print(f"User-Item Matrix Shape: {user_item_matrix_filled.shape}")
user_item_matrix_filled.head()

User-Item Matrix Shape: (943, 1682)


movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#this comutes Cosine Similarity between users
user_similarity = cosine_similarity(user_item_matrix_filled)
user_similarity_df = pd.DataFrame(user_similarity,
                                  index=user_item_matrix_filled.index,
                                  columns=user_item_matrix_filled.index)

print("User Similarity Matrix calculated.")
user_similarity_df.iloc[:5, :5]

User Similarity Matrix calculated.


user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475
2,0.166931,1.000000,0.110591,0.178121,0.072979
3,0.047460,0.110591,1.000000,0.344151,0.021245
4,0.064358,0.178121,0.344151,1.000000,0.031804
5,0.378475,0.072979,0.021245,0.031804,1.000000


In [ ]:
def get_user_recommendations(user_id, num_neighbors=5, num_recommendations=5):
    #Gets similar users sorted by similarity score (excluding the user themselves)
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).index[1:num_neighbors+1]

    #Gets movies seen by the target user
    user_seen_movies = user_item_matrix.loc[user_id].dropna().index

    #Looks at movies liked (rating >= 4) by similar users
    recommended_movies = []
    for neighbor in similar_users:
        # Finds movies neighbor rated 4 or 5 that the target user hasn't seen
        neighbor_likes = ratings[(ratings['user_id'] == neighbor) & (ratings['rating'] >= 4)]['movie_id']
        new_picks = set(neighbor_likes) - set(user_seen_movies)
        recommended_movies.extend(list(new_picks))

    #Gets the movie titles for the unique movie IDs
    top_movie_ids = pd.Series(recommended_movies).value_counts().index[:num_recommendations]

    # Maps IDs to Titles
    movie_titles = movies[movies['movie_id'].isin(top_movie_ids)]['movie_title']
    return movie_titles


target_user = 1
print(f"Recommendations for User {target_user} based on similar users:")
print(get_user_recommendations(target_user))

Recommendations for User 1 based on similar users:
285                          English Patient, The (1996)
402                                        Batman (1989)
424                                   Bob Roberts (1992)
473    Dr. Strangelove or: How I Learned to Stop Worr...
651         Rosencrantz and Guildenstern Are Dead (1990)
Name: movie_title, dtype: object


In [ ]:
target_user = 29
print(f"Recommendations for User {target_user} based on similar users:")
print(get_user_recommendations(target_user))

Recommendations for User 29 based on similar users:
288                Evita (1996)
300             In & Out (1997)
304       Ice Storm, The (1997)
321       Murder at 1600 (1997)
327    Conspiracy Theory (1997)
Name: movie_title, dtype: object


In [ ]:
target_user = 13
print(f"Recommendations for User {target_user} based on similar users:")
print(get_user_recommendations(target_user))

Recommendations for User 13 based on similar users:
133             Citizen Kane (1941)
191              Raging Bull (1980)
478                  Vertigo (1958)
489         To Catch a Thief (1955)
495    It's a Wonderful Life (1946)
Name: movie_title, dtype: object


In [ ]:
target_user = 23
print(f"Recommendations for User {target_user} based on similar users:")
print(get_user_recommendations(target_user))

Recommendations for User 23 based on similar users:
3                                Get Shorty (1995)
68                             Forrest Gump (1994)
199                            Shining, The (1980)
317                        Schindler's List (1993)
968    Winnie the Pooh and the Blustery Day (1968)
Name: movie_title, dtype: object


In [ ]:
target_user = 7
print(f"Recommendations for User {target_user} based on similar users:")
print(get_user_recommendations(target_user))

Recommendations for User 7 based on similar users:
0                  Toy Story (1995)
116                Rock, The (1996)
524           Big Sleep, The (1946)
704      Singin' in the Rain (1952)
769    Devil in a Blue Dress (1995)
Name: movie_title, dtype: object
